#### 📌 小结:把 Accessing / Updating State 串起来看

这一节信息密度比较大,这里用"问题 → 方案 → 返回值"三步把它捋一遍。

**一、问题出在哪**

之前的 `calculator(operation, a, b)` 工具,LLM 看到签名就能从用户提问里推断出三个参数。

现在我们希望工具能**读取 / 修改 graph 的 state**(把每次运算记录到 `ops` 列表里),自然想加两个参数:

```python
def calculator_wstate(operation, a, b, state, tool_call_id)
```

但 LLM **根本不知道 `state` 长什么样**,也无法生成运行时才存在的 `tool_call_id`。上面 `state_arg_diagram.png` 那张图画的就是这个困境。

**二、解决方案:`InjectedState` / `InjectedToolCallId`**

```python
state: Annotated[CalcState, InjectedState],
tool_call_id: Annotated[str, InjectedToolCallId],
```

`Annotated[..., InjectedState]` 这个标记做了两件事:

1. **对 LLM 隐藏**:生成给 LLM 看的 tool schema 时,把这两个参数**剥掉**,LLM 只看得到 `operation, a, b`。
2. **运行时注入**:`ToolNode` 真正执行工具时,LangGraph 自己把当前 state 和这次调用的 tool_call_id **填进去**。

`inject_state_diagram.png` 画的就是这个流程 —— LLM 只产生 `{operation, a, b}`,到了 ToolNode 这一层,框架再把 state / tool_call_id 注入进去。

**三、返回值为什么要用 `Command`**

之前 calculator 直接 `return result`,框架会自动把它包成 `ToolMessage` 加进 `messages`。

现在要**同时更新 `ops` 和 `messages` 两个字段**,普通 return 就不够用了,所以改成:

```python
return Command(
    update={
        "ops": ops,
        "messages": [ToolMessage(f"{result}", tool_call_id=tool_call_id)],
    }
)
```

`Command(update=...)` 等于告诉 graph:"请按这个 dict 去更新 state",然后各字段的 reducer(`add_messages`、`reduce_list`)负责把新值合并进去。

⚠️ 注意:一旦用 `Command` 自己接管返回,框架就**不会再自动包 ToolMessage** 了 —— 所以要手动造一个,并且**必须带上 `tool_call_id`**,这样 LLM 才能把结果跟它之前发出的那个 tool call 对应起来(回忆前面的 sequence diagram)。

**一句话总结**

> `InjectedState` = 让工具拿得到 graph 内部状态,但不暴露给 LLM;`Command` = 让工具一次性更新 state 里的多个字段。
